# Lesson 03 - รูปแบบการออกแบบเชิงเอเจนต์

ในบทเรียนนี้ เราจะสำรวจรูปแบบการออกแบบพื้นฐานสามอย่างสำหรับการสร้างเอเจนต์ AI ที่มีประสิทธิภาพ:

1. **คำสั่งเอเจนต์ที่ชัดเจน** — การสร้างพรอมต์ที่แม่นยำและกำหนดบทบาทเพื่อชี้นำพฤติกรรมของเอเจนต์
2. **ผลลัพธ์ที่มีโครงสร้างด้วย Pydantic Models** — การทำให้อเอเจนต์ส่งคืนข้อมูลที่คาดเดาได้และผ่านการตรวจสอบ
3. **เอเจนต์ที่มีความรับผิดชอบเดียว** — การออกแบบเอเจนต์ที่มีความเฉพาะเจาะจงและทำหน้าที่ใดหน้าที่หนึ่งได้ดี

เราจะนำแต่ละรูปแบบไปใช้กับกรณีการใช้งาน **ระบบแนะนำจุดหมายปลายทางเดินทาง** โดยค่อยๆ สร้างระบบที่สามารถแนะนำจุดหมาย ตรวจสอบความพร้อม และจัดการเรื่องโลจิสติกส์ได้


## การติดตั้ง


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: คำสั่งตัวแทนที่ชัดเจน

รูปแบบที่มีผลกระทบมากที่สุดก็คือรูปแบบที่ง่ายที่สุด: การเขียนคำสั่งที่ชัดเจนและละเอียดสำหรับตัวแทนของคุณ

คำสั่งที่ดีจะกำหนด:
- **ใคร** คือ ตัวแทน (บุคลิกและโทนเสียง)
- **อะไร** ที่ตัวแทนควรทำ (ความรับผิดชอบทีละขั้นตอน)
- **อย่างไร** ที่ตัวแทนควรปฏิบัติตัว (ข้อจำกัดและสไตล์)

ด้านล่างนี้ เราสร้างตัวแทนผู้ช่วยท่องเที่ยวด้วยคำสั่งที่ชัดเจนซึ่งกำหนดทิศทางทุกคำตอบที่มันผลิตขึ้นมา


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: โครงสร้างผลลัพธ์ด้วยโมเดล Pydantic

ข้อความรูปแบบอิสระมีประโยชน์สำหรับการสนทนา แต่ระบบ downstream ต้องการข้อมูลที่มีโครงสร้าง  
โดยการจับคู่ **โมเดล Pydantic** กับ **ฟังก์ชันเครื่องมือ** เราสามารถ:

- กำหนดสกีมาเฉพาะสำหรับผลลัพธ์ของเอเย่นต์  
- ตรวจสอบความถูกต้องของคำตอบโดยอัตโนมัติ  
- ผสานผลลัพธ์ของเอเย่นต์เข้ากับตรรกะของแอปพลิเคชันอย่างน่าเชื่อถือ  

เรายังแนะนำเครื่องมือที่ส่งคืนรายละเอียดปลายทางเพื่อให้เอเย่นต์ตั้งอยู่บนคำแนะนำที่อิงข้อมูลจริงด้วยอีกด้วย


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## รูปแบบที่ 3: ตัวแทนความรับผิดชอบเดี่ยว

งานที่ซับซ้อนจะได้ประโยชน์จากการแบ่งงานออกเป็นตัวแทนที่มีความเชี่ยวชาญเฉพาะด้านแต่ละคน โดยแต่ละคนมีความรับผิดชอบเพียงอย่างเดียว:

- **ผู้เชี่ยวชาญปลายทาง** ที่รู้เกี่ยวกับสถานที่และความพร้อมใช้งาน
- **ผู้วางแผนโลจิสติกส์** ที่จัดการเที่ยวบิน โรงแรม และตารางเดินทาง

สิ่งนี้สะท้อนหลักการวิศวกรรมซอฟต์แวร์เรื่อง *การแยกความรับผิดชอบ* — ตัวแทนแต่ละตัวจึงง่ายต่อการทดสอบ ดูแลรักษา และพัฒนาแยกกันได้อย่างอิสระ


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## สรุป

ในบทเรียนนี้เราได้นำรูปแบบการออกแบบสามแบบที่เน้นตัวแทนมาใช้กับสถานการณ์แนะนำการเดินทาง:

| รูปแบบ | แนวคิดหลัก | ประโยชน์ |
|---|---|---|
| **คำแนะนำที่ชัดเจน** | กำหนดบุคลิกภาพ ความรับผิดชอบ และข้อจำกัดล่วงหน้า | พฤติกรรมตัวแทนที่สอดคล้องและตรงกับแบรนด์ |
| **ผลลัพธ์ที่มีโครงสร้าง** | ใช้โมเดล Pydantic เป็นรูปแบบการตอบกลับ | ผลลัพธ์ที่ได้รับการตรวจสอบและเครื่องอ่านได้ |
| **ความรับผิดชอบเดียว** | มอบงานที่เน้นเฉพาะให้ตัวแทนแต่ละตัว | ง่ายต่อการทดสอบ ดูแลรักษา และประสานงาน |

รูปแบบเหล่านี้สามารถประสานกันได้อย่างเป็นธรรมชาติ — คุณสามารถรวมคำแนะนำที่ชัดเจนกับผลลัพธ์ที่มีโครงสร้างภายในตัวแทนที่มีความรับผิดชอบเดียวเพื่อสร้างระบบที่แข็งแกร่งและพร้อมใช้งานในสภาพแวดล้อมการผลิตได้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้มีความถูกต้องสูงสุด โปรดทราบว่าการแปลด้วยระบบอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาดั้งเดิมถือเป็นแหล่งข้อมูลที่น่าเชื่อถือที่สุด สำหรับข้อมูลที่สำคัญ ควรใช้บริการแปลโดยผู้เชี่ยวชาญทางภาษาเป็นทางเลือก เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
